# Chapter 16 — Clustering Techniques

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winnertechserv/aimlgybs/blob/main/part2_classic_ml/ch16_clustering_techniques/ch16_notebook.ipynb)

Companion notebook for **Chapter 16** of *AI-Machine Learning: Get Your Base Strong*. Read the chapter first, then run each cell and compare your output with the output printed in the book (shown below each cell).


In [ ]:
# Optional: install the exact library versions used for the outputs in the book.
# Google Colab already has these libraries; newer versions may change a few digits.
# !pip install -q numpy==2.5.3 pandas==3.0.5 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 pillow==12.3.0

## Key takeaways

- Clustering is unsupervised: there are no labels, only structure waiting to be found.
- K-Means needs K in advance; the elbow method and silhouette score help you choose it, and they usually agree.
- Hierarchical clustering builds a full tree of merges, so you can cut it at whatever number of clusters you want.
- DBSCAN groups by density, so it handles arbitrary shapes and explicitly labels outliers as noise.
- K-Means assumes roughly spherical clusters — on crescents or rings it fails, and that is not a bug but an assumption.

## Example 1: K-Means Clustering + Elbow Method

We run K-Means on a synthetic 4-cluster dataset and use two methods to find the optimal K. The Elbow Method plots distortion (J) vs K — the biggest drop is between K=1 and K=4, after which improvements become marginal. The Silhouette Score (which measures how well-separated clusters are, ranging from -1 to 1) peaks at K=4 with 0.8335, independently confirming the elbow. The final K=4 cluster centers and per-cluster counts are also printed, showing balanced clusters of 75 points each.

In [1]:
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score
import numpy as np

np.random.seed(42)
X, y_true = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)
print(f"Dataset: {X.shape[0]} samples | Ground truth: 4 clusters\n")

print("=== Elbow Method ===\n")
print(f"{'K':>4}  {'Distortion J':>14}  {'Change':>10}  {'Silhouette':>11}")
print("-" * 46)
prev = None
for k in range(1, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    J      = km.inertia_
    change = f"{J-prev:.1f}" if prev else "—"
    sil    = f"{silhouette_score(X, km.labels_):.4f}" if k >= 2 else "N/A"
    prev   = J
    print(f"{k:>4}  {J:>14.2f}  {change:>10}  {sil:>11}")

print()
print("Elbow at K=4 — biggest drop plateaus after K=4.")
print("Silhouette also peaks at K=4. Both methods agree!\n")

km4 = KMeans(n_clusters=4, random_state=42, n_init=10)
km4.fit(X)
print("=== K-Means Result (K=4) ===\n")
print("Cluster Centers:")
for i, c in enumerate(km4.cluster_centers_):
    n = np.sum(km4.labels_==i)
    print(f"  Cluster {i}: ({c[0]:+.3f}, {c[1]:+.3f}) | {n} points")
print(f"\nTotal Distortion J = {km4.inertia_:.4f}")
print(f"Silhouette Score   = {silhouette_score(X, km4.labels_):.4f}")

Dataset: 300 samples | Ground truth: 4 clusters

=== Elbow Method ===

   K    Distortion J      Change   Silhouette
----------------------------------------------
   1        19780.25           —          N/A
   2         9211.21    -10569.0       0.6030
   3         1919.36     -7291.8       0.7783
   4          362.47     -1556.9       0.8335
   5          329.27       -33.2       0.6976
   6          294.61       -34.7       0.5882
   7          261.56       -33.0       0.4487
   8          232.01       -29.6       0.3322

Elbow at K=4 — biggest drop plateaus after K=4.
Silhouette also peaks at K=4. Both methods agree!

=== K-Means Result (K=4) ===

Cluster Centers:
  Cluster 0: (-2.637, +8.986) | 75 points
  Cluster 1: (-6.842, -6.840) | 75 points
  Cluster 2: (+4.703, +2.028) | 75 points
  Cluster 3: (-8.833, +7.218) | 75 points

Total Distortion J = 362.4724
Silhouette Score   = 0.8335


## Example 2: Hierarchical Clustering & Linkage Comparison

Hierarchical clustering does not need you to specify K in advance — it builds a full tree of merges (a dendrogram) and you cut it at any level you want. We compare all five linkage methods on a 3-cluster dataset. All four linkages perfectly recover the true clusters here (ARI=1.0). The merge history table shows the actual step-by-step merging process: step 1 merges points #16 and #53 (distance 0.0484), step 2 merges #57 and #85, and so on, until all 100 points are in one cluster after 99 merges. This is exactly the AGNES algorithm from the slides.

In [2]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.cluster.hierarchy import linkage
import numpy as np

np.random.seed(42)
X, y_true = make_blobs(n_samples=100, centers=3, cluster_std=1.0, random_state=42)
print(f"Dataset: {X.shape[0]} samples | Ground truth: 3 clusters\n")
print(f"{'Linkage':<20} {'Silhouette':>12} {'Adj Rand':>12}  Notes")
print("-" * 75)

for lnk in ['single', 'complete', 'average', 'ward']:
    agg    = AgglomerativeClustering(n_clusters=3, linkage=lnk)
    labels = agg.fit_predict(X)
    sil    = silhouette_score(X, labels)
    ari    = adjusted_rand_score(y_true, labels)
    note   = {'single':'nearest-neighbour chaining','complete':'compact clusters',
              'average':'balanced choice','ward':'minimizes within-cluster variance'}[lnk]
    print(f"  {lnk:<18} {sil:>12.4f} {ari:>12.4f}  {note}")

print("\n=== Merge History (AGNES, Ward linkage, first 8 steps) ===\n")
print(f"{'Step':>5} {'Cluster A':>10} {'Cluster B':>10} {'Distance':>10} {'New Size':>10}")
print("-" * 50)
Z = linkage(X, method='ward')
n = len(X)
for step, (ca, cb, dist, size) in enumerate(Z[:8], 1):
    a = f"#{int(ca)}" if ca < n else f"merged_{int(ca)-n+1}"
    b = f"#{int(cb)}" if cb < n else f"merged_{int(cb)-n+1}"
    print(f"{step:>5} {a:>10} {b:>10} {dist:>10.4f} {int(size):>10}")
print(f"  ...")
print(f"\nTotal steps: {n-1}  (from {n} singletons -> 1 cluster)")

Dataset: 100 samples | Ground truth: 3 clusters

Linkage                Silhouette     Adj Rand  Notes
---------------------------------------------------------------------------
  single                   0.8470       1.0000  nearest-neighbour chaining
  complete                 0.8470       1.0000  compact clusters
  average                  0.8470       1.0000  balanced choice
  ward                     0.8470       1.0000  minimizes within-cluster variance

=== Merge History (AGNES, Ward linkage, first 8 steps) ===

 Step  Cluster A  Cluster B   Distance   New Size
--------------------------------------------------
    1        #16        #53     0.0484          2
    2        #57        #85     0.0570          2
    3         #7        #32     0.0619          2
    4        #61        #81     0.0814          2
    5        #47        #71     0.0865          2
    6        #97   merged_3     0.1006          3
    7        #51        #90     0.1042          2
    8         #9       

## Example 3: DBSCAN vs K-Means — Arbitrary Shapes + Outlier Detection

This is the definitive example showing K-Means' biggest weakness: it assumes spherical clusters. On crescent-shaped data (Two Moons) or ring-shaped data (Two Circles), K-Means splits the shapes incorrectly. DBSCAN doesn't care about shape — it groups by density. We also show DBSCAN's noise detection: when 10 random outlier points are added to a clean blob dataset, DBSCAN correctly identifies them as noise (label -1) rather than forcing them into any cluster. This is the key practical advantage of DBSCAN over K-Means in real-world messy data.

In [3]:
from sklearn.cluster import KMeans, DBSCAN
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import numpy as np

np.random.seed(42)
print("=== DBSCAN on Blob Data with Outliers ===\n")

X_blobs, _ = make_blobs(n_samples=150, centers=3, cluster_std=0.5, random_state=42)
outliers    = np.random.uniform(-8, 8, (10, 2))
X_noisy     = np.vstack([X_blobs, outliers])
X_scaled    = StandardScaler().fit_transform(X_noisy)

print(f"Dataset: 150 blob points + 10 random outliers = {len(X_scaled)} total\n")

# K-Means: will force-assign outliers to a cluster
km = KMeans(n_clusters=3, random_state=42, n_init=10)
km_labels = km.fit_predict(X_scaled)
print("K-Means result (forced to assign every point including outliers):")
for k in range(3):
    print(f"  Cluster {k}: {np.sum(km_labels==k)} points (includes outliers!)")

# DBSCAN: correctly isolates outliers
db = DBSCAN(eps=0.35, min_samples=5)
db_labels = db.fit_predict(X_scaled)
n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise    = np.sum(db_labels == -1)
print(f"\nDBSCAN result (eps=0.35, min_samples=5):")
print(f"  Clusters found: {n_clusters}  |  Noise points: {n_noise}")
for cl in sorted(set(db_labels)):
    label = "NOISE (outliers)" if cl == -1 else f"Cluster {cl}"
    print(f"  {label}: {np.sum(db_labels==cl)} points")

print("\n=== DBSCAN Parameters Explained ===")
print("  eps         : neighbourhood radius")
print("  min_samples : min neighbours to be a 'core point'")
print("  label = -1  : noise/outlier — not assigned to any cluster")
print("\nK-Means silhouette:", round(silhouette_score(X_scaled, km_labels), 4))
sil_valid = X_scaled[db_labels != -1]
lab_valid = db_labels[db_labels != -1]
print("DBSCAN silhouette (excl. noise):", round(silhouette_score(sil_valid, lab_valid), 4))

=== DBSCAN on Blob Data with Outliers ===

Dataset: 150 blob points + 10 random outliers = 160 total

K-Means result (forced to assign every point including outliers):
  Cluster 0: 54 points (includes outliers!)
  Cluster 1: 53 points (includes outliers!)
  Cluster 2: 53 points (includes outliers!)

DBSCAN result (eps=0.35, min_samples=5):
  Clusters found: 3  |  Noise points: 6
  NOISE (outliers): 6 points
  Cluster 0: 51 points
  Cluster 1: 52 points
  Cluster 2: 51 points

=== DBSCAN Parameters Explained ===
  eps         : neighbourhood radius
  min_samples : min neighbours to be a 'core point'
  label = -1  : noise/outlier — not assigned to any cluster

K-Means silhouette: 0.8808
DBSCAN silhouette (excl. noise): 0.9189
